In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
df_health = spark.read.csv(
    "Files/raw/healthcare_dataset.csv",
    header=True,
    inferSchema=True
)

df_disease = spark.read.csv(
    "Files/raw/api_disease_data.csv",
    header=True,
    inferSchema=True
)

display(df_health.limit(5))

StatementMeta(, 4464f8bc-d59d-432b-b358-4ff3a9eafb60, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 89c7081e-0125-47e7-858d-c42f25ffce45)

In [2]:
from pyspark.sql import functions as F

df_cleaned = df_health \
    .withColumn("Name", F.initcap(F.col("Name"))) \
    .dropDuplicates() \
    .dropna()

StatementMeta(, 4464f8bc-d59d-432b-b358-4ff3a9eafb60, 4, Finished, Available, Finished, False)

In [3]:
df_cleaned = df_cleaned \
    .withColumn(
        "Length_of_Stay",
        F.datediff(
            F.col("Discharge Date"),
            F.col("Date of Admission")
        )
    )

StatementMeta(, 4464f8bc-d59d-432b-b358-4ff3a9eafb60, 5, Finished, Available, Finished, False)

In [4]:
df_cleaned = df_cleaned.withColumn(
    "Age_Group",
    F.when(F.col("Age") < 35, "Young")
     .when(F.col("Age") < 60, "Middle")
     .otherwise("Senior")
)

StatementMeta(, 4464f8bc-d59d-432b-b358-4ff3a9eafb60, 6, Finished, Available, Finished, False)

In [5]:
new_cols = [c.replace(" ", "_") for c in df_cleaned.columns]

df_cleaned = df_cleaned.toDF(*new_cols)

StatementMeta(, 4464f8bc-d59d-432b-b358-4ff3a9eafb60, 7, Finished, Available, Finished, False)

In [6]:
df_cleaned.write \
    .mode("overwrite") \
    .format("delta") \
    .save("Tables/healthcare_silver")

StatementMeta(, 4464f8bc-d59d-432b-b358-4ff3a9eafb60, 8, Finished, Available, Finished, False)

In [7]:
df_silver = spark.read.format("delta").load("Tables/healthcare_silver")

display(df_silver.limit(5))

StatementMeta(, 4464f8bc-d59d-432b-b358-4ff3a9eafb60, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f25c0322-cb8b-4959-a8a4-eae6fb1e344a)